# Maryland Hispanic-Origin Dot Distribution Map

This notebook recreates the style of the supplied multivariate dot-density map for Maryland. Each dot represents approximately the same number of people, and color identifies the origin group.

**Groups:** Mexican, Puerto Rican, Cuban, Dominican, Guatemalan, Honduran, Nicaraguan, Salvadoran, Colombian, and Venezuelan.

**Important:** ACS origin estimates are tract-level. Dots are randomly sampled within tract polygons and do not show actual residential locations.

## 1. Install and import packages

These packages retrieve Census data, read geographic boundaries, perform spatial operations, and draw the map. Run the install command once in your Python environment.

In [ ]:
%pip install geopandas matplotlib numpy pandas requests shapely

from pathlib import Path
from getpass import getpass
import zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import Point

## 2. Configure the map

`DOT_VALUE` determines how many people one dot represents. The colors use selected Okabe-Ito palette colors and are paired with text labels in the legend.

In [ ]:
STATE_FIPS = "24"
ACS_YEAR = 2024
DOT_VALUE = 100
RANDOM_SEED = 42

GROUPS = {
    "Mexican": "B03001_004E",
    "Puerto Rican": "B03001_005E",
    "Cuban": "B03001_006E",
    "Dominican": "B03001_007E",
    "Guatemalan": "B03001_010E",
    "Honduran": "B03001_011E",
    "Nicaraguan": "B03001_012E",
    "Salvadoran": "B03001_014E",
    "Colombian": "B03001_020E",
    "Venezuelan": "B03001_025E",
}

GROUP_COLORS = {
    "Mexican": "#4477AA",
    "Puerto Rican": "#EE6677",
    "Cuban": "#228833",
    "Dominican": "#CCBB44",
    "Guatemalan": "#66CCEE",
    "Honduran": "#AA3377",
    "Nicaraguan": "#BBBBBB",
    "Salvadoran": "#332288",
    "Colombian": "#44AA99",
    "Venezuelan": "#DDCC77",
}

MAP_CRS = "EPSG:26918"
DATA_DIR = Path("census_data")
OUTPUT_DIR = Path("dot_distribution_output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

ACS_API_URL = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"
TRACT_ZIP_URL = (
    "https://www2.census.gov/geo/tiger/TIGER2024/TRACT/"
    f"tl_2024_{STATE_FIPS}_tract.zip"
)

## 3. Prompt for a Census API key

The key is optional for this one API request. `getpass()` hides the input while typing, and pressing Enter without a key is supported.

In [ ]:
print("Census API key is optional for this single request.")
print("Request a free key at https://api.census.gov/data/key_signup.html")
api_key = getpass("Enter API key, or press Enter to continue without one: ").strip()

## 4. Retrieve 2024 ACS tract estimates

Table B03001 contains Hispanic or Latino origin by specific origin. The `E` variables are estimates; corresponding `M` variables are margins of error. The GEOID is assembled from state, county, and tract codes so the data can be joined to TIGER/Line.

In [ ]:
estimate_variables = list(GROUPS.values())
moe_variables = [v.replace("E", "M") for v in estimate_variables]
params = {
    "get": ",".join(["NAME"] + estimate_variables + moe_variables),
    "for": "tract:*",
    "in": f"state:{STATE_FIPS}",
}
if api_key:
    params["key"] = api_key

response = requests.get(ACS_API_URL, params=params, timeout=120)
response.raise_for_status()
rows = response.json()
acs_data = pd.DataFrame(rows[1:], columns=rows[0])
for col, width in {"state": 2, "county": 3, "tract": 6}.items():
    acs_data[col] = acs_data[col].astype(str).str.zfill(width)
acs_data["GEOID"] = acs_data["state"] + acs_data["county"] + acs_data["tract"]
for col in estimate_variables + moe_variables:
    acs_data[col] = pd.to_numeric(acs_data[col], errors="coerce")
acs_data[estimate_variables] = acs_data[estimate_variables].fillna(0).clip(lower=0)
print(f"Retrieved {len(acs_data):,} Maryland tracts.")
acs_data.head()

## 5. Download and read tract boundaries

TIGER/Line supplies tract geometry and geographic identifiers. It does not contain the ACS population estimates, so the two datasets are joined in the next step.

In [ ]:
zip_path = DATA_DIR / f"tl_2024_{STATE_FIPS}_tract.zip"
extract_dir = DATA_DIR / f"tl_2024_{STATE_FIPS}_tract"
shapefile = extract_dir / f"tl_2024_{STATE_FIPS}_tract.shp"
if not shapefile.exists():
    if not zip_path.exists():
        r = requests.get(TRACT_ZIP_URL, timeout=300)
        r.raise_for_status()
        zip_path.write_bytes(r.content)
    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(extract_dir)

tracts = gpd.read_file(shapefile)
tracts["GEOID"] = tracts["GEOID"].astype(str).str.zfill(11)
tracts["COUNTYFP"] = tracts["COUNTYFP"].astype(str).str.zfill(3)
tracts = tracts[["GEOID", "COUNTYFP", "NAME", "geometry"]].copy()
print(f"Loaded {len(tracts):,} tract polygons.")

## 6. Join estimates to geography and project the map

The GEOID join links each tract estimate to the correct polygon. EPSG:26918 is a projected coordinate reference system appropriate for Maryland; it uses meters rather than degrees.

In [ ]:
map_data = tracts.merge(acs_data, on="GEOID", how="inner")
map_data = map_data.loc[map_data.geometry.notna() & (~map_data.geometry.is_empty)].copy()
if map_data.empty:
    raise ValueError("The ACS-to-tract join returned no records.")
map_data = map_data.to_crs(MAP_CRS)
print(f"Matched {len(map_data):,} tracts.")

## 7. Generate dots for each subgroup

The code samples points inside each tract polygon. Stochastic rounding converts a fractional expected dot count into a whole number without always rounding down. Because the source estimates are tract-level, the within-tract pattern is illustrative rather than observed.

In [ ]:
def random_points_in_polygon(polygon, count, rng):
    """Sample random points inside a tract polygon."""
    if count <= 0:
        return []
    min_x, min_y, max_x, max_y = polygon.bounds
    points, attempts = [], 0
    max_attempts = max(1000, count * 100)
    while len(points) < count and attempts < max_attempts:
        point = Point(rng.uniform(min_x, max_x), rng.uniform(min_y, max_y))
        if polygon.contains(point):
            points.append(point)
        attempts += 1
    if len(points) < count:
        fallback = polygon.representative_point()
        points.extend([fallback] * (count - len(points)))
    return points

rng = np.random.default_rng(RANDOM_SEED)
dot_layers = {}
for group_name, variable in GROUPS.items():
    xs, ys = [], []
    for row in map_data.itertuples():
        value = getattr(row, variable)
        value = max(0, float(value)) if pd.notna(value) else 0
        expected = value / DOT_VALUE
        whole = int(np.floor(expected))
        count = whole + int(rng.random() < (expected - whole))
        for point in random_points_in_polygon(row.geometry, count, rng):
            xs.append(point.x)
            ys.append(point.y)
    dot_layers[group_name] = (np.asarray(xs), np.asarray(ys))
    print(f"{group_name}: {len(xs):,} dots")

## 8. Draw and save the map

All categories use the same dot size. Color distinguishes the four origin groups; the written legend labels each group and explains the dot value. County outlines provide orientation.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 15), facecolor="white")
ax.set_facecolor("#EAF3F8")
county_boundaries = map_data.dissolve(by="COUNTYFP")
county_boundaries.boundary.plot(
    ax=ax, color="#4B5563", linewidth=0.9, zorder=2
)

# County labels are placed at representative points inside each county.
county_names = {
    "001": "Allegany", "003": "Anne Arundel", "005": "Baltimore County",
    "009": "Calvert", "011": "Caroline", "013": "Carroll",
    "015": "Cecil", "017": "Charles", "019": "Dorchester",
    "021": "Frederick", "023": "Garrett", "025": "Harford",
    "027": "Howard", "029": "Kent", "031": "Montgomery",
    "033": "Prince George's", "035": "Queen Anne's", "037": "St. Mary's",
    "039": "Somerset", "041": "Talbot", "043": "Washington",
    "045": "Wicomico", "047": "Worcester", "510": "Baltimore City",
}
import matplotlib.patheffects as path_effects
for county_fips, county_row in county_boundaries.iterrows():
    label = county_names.get(str(county_fips).zfill(3))
    if label:
        point = county_row.geometry.representative_point()
        ax.annotate(
            label, xy=(point.x, point.y), ha="center", va="center",
            fontsize=7.0, fontweight="bold", color="#17212B", zorder=8,
            bbox={"boxstyle": "round,pad=0.22", "facecolor": "white",
                  "edgecolor": "#687482", "linewidth": 0.45, "alpha": 0.94},
            path_effects=[path_effects.withStroke(
                linewidth=2.2, foreground="white")]
        )
for group_name, (xs, ys) in dot_layers.items():
    ax.scatter(xs, ys, s=4.0, marker="o", color=GROUP_COLORS[group_name],
               alpha=1.0, linewidths=0, edgecolors="none",
               antialiaseds=True, rasterized=False, label=group_name, zorder=3)
ax.set_title(
    "U.S. CENSUS BUREAU | MARYLAND HISPANIC-ORIGIN POPULATION",
    fontsize=17, fontweight="bold", loc="left", pad=24,
)
ax.text(
    0, 1.018,
    f"OFFICIAL SOURCE: {ACS_YEAR} AMERICAN COMMUNITY SURVEY (ACS) 5-YEAR ESTIMATES",
    transform=ax.transAxes, fontsize=10.5, fontweight="bold",
    color="#17212B", ha="left",
)
ax.text(
    0, 1.002,
    f"GEOGRAPHY: MARYLAND CENSUS TRACTS • EACH DOT ≈ "
    f"{DOT_VALUE:,} PEOPLE IN THE SELECTED ORIGIN GROUP",
    transform=ax.transAxes, fontsize=9.5, color="#404040", ha="left",
)
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker="o", linestyle="None",
           markerfacecolor=GROUP_COLORS[name], markeredgecolor="#263238",
           markeredgewidth=0.35, markersize=9.5, label=name)
    for name in GROUPS
]
legend = ax.legend(handles=legend_handles,
                   title=f"ACS estimated population\\nEach dot ≈ {DOT_VALUE:,} people",
                   loc="lower left", ncol=2, columnspacing=1.1,
                   handletextpad=0.5, borderpad=0.9, labelspacing=0.65,
                   frameon=True, framealpha=0.99, facecolor="white",
                   edgecolor="#4B5563", fontsize=9.5, title_fontsize=10)
legend.set_zorder(10)
ax.set_aspect("equal")
ax.axis("off")
fig.text(
    0.07, 0.055,
    f"DATA SOURCE: U.S. CENSUS BUREAU | {ACS_YEAR} AMERICAN COMMUNITY SURVEY (ACS) 5-YEAR ESTIMATES",
    fontsize=9.2, fontweight="bold", color="#17212B", ha="left",
)
fig.text(
    0.07, 0.033,
    "TABLE B03001: HISPANIC OR LATINO ORIGIN BY SPECIFIC ORIGIN | GEOGRAPHY: CENSUS TRACTS | BOUNDARIES: 2024 TIGER/LINE",
    fontsize=8.0, color="#333333", ha="left",
)
fig.text(
    0.07, 0.012,
    "Dots are randomly distributed within tract boundaries and do not show actual residences. ACS values are estimates with sampling error.",
    fontsize=7.6, color="#333333", ha="left",
)
plt.tight_layout(rect=[0.03, 0.11, 0.98, 0.94])
output_path = OUTPUT_DIR / "maryland_hispanic_origin_dot_distribution.png"
pdf_path = OUTPUT_DIR / "maryland_hispanic_origin_dot_distribution.pdf"
fig.savefig(output_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)
print(f"High-resolution PNG saved to: {output_path.resolve()}")
print(f"Vector PDF saved to: {pdf_path.resolve()}")

## 9. Interpretation and limitations

- Dots are randomly placed within census tracts; they do not identify actual homes or addresses.
- Tracts can contain parks, industrial land, and water. This example does not constrain dots to residential land.
- ACS 5-year estimates have sampling error. Review the reported margins of error before comparing small differences.
- Stochastic rounding means dot counts approximate each tract estimate divided by the dot value.
- Do not interpret the random within-tract pattern as evidence of neighborhood-level clustering.

## Data sources

- [2024 ACS 5-Year Table B03001 variables](https://api.census.gov/data/2024/acs/acs5/groups/B03001.html)
- [2024 ACS 5-Year data](https://www.census.gov/data/developers/data-sets/acs-5year.html)
- [2024 Maryland TIGER/Line Census Tracts](https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_24_tract.zip)
- [Census API key signup](https://api.census.gov/data/key_signup.html)
